In [3]:
import re
import os
from langdetect import detect, DetectorFactory
import nltk
from nltk.corpus import stopwords

# Ensures consistent results for language detection
DetectorFactory.seed = 0

# Download stopwords (only happens once)
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def clean_txt_file(input_filename, output_filename):
    if not os.path.exists(input_filename):
        print(f"Error: {input_filename} not found.")
        return

    with open(input_filename, 'r', encoding='utf-8') as f:
        content = f.read()

    # 1. Split by page separators (===)
    entries = re.split(r'={10,}', content)
    
    cleaned_paragraphs = []

    for entry in entries:
        lines = entry.split('\n')
        page_text_parts = []
        
        for line in lines:
            # Basic cleanup and lowercase conversion
            clean_line = line.strip().lower()
            
            # 2. Skip empty lines
            if not clean_line:
                continue
            
            # 3. Skip Metadata and specific noise
            if any(meta in clean_line for meta in ['url:', 'title:', 'words:', '_redirecttologinpage_']):
                continue
                
            # 4. Filter: STARTING with 'pdf' OR CONTAINING 'error'
            if clean_line.startswith('pdf') or 'error' in clean_line:
                continue
            
            # 5. Remove numbers and special characters (keep only letters and spaces)
            clean_line = re.sub(r'[^a-z\s]', '', clean_line)
            
            # 6. Stopword Removal
            words = clean_line.split()
            filtered_words = [w for w in words if w not in stop_words]
            final_line = " ".join(filtered_words)
            
            # 7. Length check (filters out tiny menu fragments)
            if len(final_line) > 20: 
                page_text_parts.append(final_line)

        # Combine lines from the same entry
        full_paragraph = " ".join(page_text_parts).strip()

        # 8. Final English Language Filter
        if full_paragraph:
            try:
                if detect(full_paragraph) == 'en':
                    cleaned_paragraphs.append(full_paragraph)
            except:
                continue

    # 9. Remove Duplicates (keeps order)
    seen = set()
    unique_paragraphs = [p for p in cleaned_paragraphs if not (p in seen or seen.add(p))]

    # 10. Write to output file
    with open(output_filename, 'w', encoding='utf-8') as out_f:
        for p in unique_paragraphs:
            out_f.write(p + "\n\n")
    
    print("-" * 30)
    print(f"Cleaning Process Complete!")
    print(f"Total entries processed: {len(entries)}")
    print(f"Unique English paragraphs saved: {len(unique_paragraphs)}")
    print(f"Output file: {output_filename}")
    print("-" * 30)

input_file = "/Users/jahanvigajera/Desktop/Assignment/NLU_assignment/Assignment2/problem1/iitj_complete_data/all_text.txt"  # Ensure your source file is named correctly
output_file = "cleaned_corpus.txt"

clean_txt_file(input_file, output_file)

------------------------------
Cleaning Process Complete!
Total entries processed: 23321
Unique English paragraphs saved: 205
Output file: cleaned_corpus.txt
------------------------------
